# ADNI demographic and clinical preprocessing

**Thesis workflow — preprocessing only**

This notebook creates clean demographic/clinical tables before any MRI merge or machine-learning model. It deliberately does **not** impute, scale, encode, select features, split subjects, or train a model. Those operations belong inside the later training pipeline so information from validation/test subjects cannot leak into training.

| Output object | Unit of analysis | Intended use |
|---|---|---|
| `clinical_long_clean` | source record | Longitudinal outcomes and later visit alignment; released only when its expanded key is unique |
| `baseline_all` | one row per source subject | Full subject inventory with baseline-eligibility and QC flags |
| `baseline_5` | one row per subject | Exploratory 5-group entry-cohort analysis |
| `baseline_3` | one row per subject | Primary CN/MCI/AD baseline analysis |
| `baseline_model_3group` / `baseline_model_5group` | one row per subject | Inputs for the next modeling notebook; missingness retained |

**Label rule:** `entry_group_5` and `entry_group_3` describe the participant's entry cohort. `diagnosis_label` is visit-specific and is retained for quality control and future longitudinal outcomes. Do not treat `entry_research_group` as a visit-varying diagnosis. For cross-phase harmonization, raw `SMC` is preserved in `entry_group_raw` and explicitly mapped to `CN`; all raw-to-derived counts are exported.


## 1. Setup and reproducible paths

Set `ADNI_CLINICAL_CSV` if the CSV is not in the current working directory. For example, in macOS/Linux before starting Jupyter:

```bash
export ADNI_CLINICAL_CSV="/path/to/all_sub.csv"
```

Alternatively, directly edit `DATA_PATH` below. Each execution writes to a unique run folder under `OUTPUT_DIR`, so a blocked artifact cannot be confused with a clean file left by an earlier run.


In [ ]:
from pathlib import Path
from itertools import combinations
import hashlib
import os
import platform
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
try:
    from IPython.display import display
except ImportError:
    display = print
from scipy.stats import chi2_contingency, kruskal, mannwhitneyu

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")

DATA_PATH = Path(
    os.getenv("ADNI_CLINICAL_CSV", "all_sub.csv")
).expanduser()
OUTPUT_DIR = Path(
    os.getenv("ADNI_OUTPUT_DIR", "adni_preprocessed")
).expanduser()
WORKFLOW_VERSION = "2.0"

print(f"Input:  {DATA_PATH.resolve()}")
print(f"Output: {OUTPUT_DIR.resolve()}")


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Set ADNI_CLINICAL_CSV or edit DATA_PATH."
    )

df_raw = pd.read_csv(DATA_PATH, low_memory=False)
df_raw_snapshot = df_raw.copy(deep=True)
run_timestamp_utc = pd.Timestamp.now(tz="UTC")
input_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
RUN_ID = (
    run_timestamp_utc.strftime("%Y%m%dT%H%M%S%fZ")
    + f"_{input_sha256[:8]}"
)
RUN_OUTPUT_DIR = OUTPUT_DIR / RUN_ID

run_manifest = pd.DataFrame({
    "field": [
        "workflow_version", "run_timestamp_utc", "input_path",
        "input_sha256", "run_output_dir", "python_version", "platform",
        "pandas_version", "numpy_version",
    ],
    "value": [
        WORKFLOW_VERSION,
        run_timestamp_utc.isoformat(),
        str(DATA_PATH.resolve()),
        input_sha256,
        str(RUN_OUTPUT_DIR.resolve()),
        sys.version.split()[0],
        platform.platform(),
        pd.__version__,
        np.__version__,
    ],
})

print(f"Raw shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]:,} columns")
print(f"This run will export to: {RUN_OUTPUT_DIR.resolve()}")
display(df_raw.head())


## 2. Preprocessing definitions

The function below is the single source of truth for cleaning. Important decisions:

- The original dataframe is never modified.
- No source row is removed automatically. Exact duplicates and non-unique longitudinal keys are audited while `_source_row_id` and available `EXAMDATE`, `VISCODE2`, and phase/protocol fields are retained.
- `m6` is standardized to `m06`; `scmri`, unscheduled visits, and other visit codes remain available in the longitudinal table.
- Rows without a visit code are excluded from the visit-level table and recorded in an exclusions table; subjects without `sc`/`bl` remain visible in `baseline_all` with an eligibility flag.
- Baseline clinical values prefer `bl`; when absent, they fall back to `sc`. Every value receives a `source_*` column (for example, `source_CDRSB`).
- Score-range checks flag possible problems; they do not delete or winsorize observations.
- Invalid non-null exam dates block longitudinal release, and ambiguous static values are never resolved by choosing an arbitrary row.
- Model quarantine is specific to the exported primary predictors; a QC issue in an unused score remains audited but does not remove that subject from the primary model table.


In [ ]:
CORE_REQUIRED_COLUMNS = ["subject_id", "visit"]

EXPECTED_COLUMNS = [
    "subject_id", "visit", "PTGENDER", "entry_age", "entry_date",
    "entry_research_group", "GENOTYPE", "PTEDUCAT", "DIAGNOSIS",
    "CDRSB", "LDELTOTAL", "MMSCORE", "COPYSCOR", "LIMMTOTAL",
    "GDTOTAL", "FAQTOTAL", "TOTAL13",
]

OPTIONAL_RECORD_METADATA = [
    "EXAMDATE", "VISCODE2", "Phase", "PHASE", "COLPROT",
    "ORIGPROT", "source_table", "SOURCE_TABLE",
]

CLINICAL_COLS = [
    "CDRSB", "LDELTOTAL", "MMSCORE", "COPYSCOR",
    "LIMMTOTAL", "GDTOTAL", "FAQTOTAL", "TOTAL13",
]

NUMERIC_COLS = [
    "PTGENDER", "entry_age", "PTEDUCAT", "DIAGNOSIS", *CLINICAL_COLS
]

ENTRY_GROUP_5_LEVELS = ["CN", "EMCI", "LMCI", "MCI", "AD"]
ENTRY_GROUP_3_LEVELS = ["CN", "MCI", "AD"]
ENTRY_GROUP_RAW_LEVELS = ["CN", "SMC", "EMCI", "LMCI", "MCI", "AD"]

DIAGNOSIS_QC_MAP = {
    1.0: "CN",
    2.0: "MCI",
    3.0: "Dementia",
    10.0: "TEAM_NODX",
}

APOE4_DOSAGE_MAP = {
    "2/2": 0,
    "2/3": 0,
    "3/3": 0,
    "2/4": 1,
    "3/4": 1,
    "4/4": 2,
}

# Plausible instrument limits used only as QC flags. Confirm variable
# definitions against the ADNI data dictionary for the downloaded table.
SCORE_LIMITS = {
    "CDRSB": (0, 18),
    "LDELTOTAL": (0, 25),
    "MMSCORE": (0, 30),
    "COPYSCOR": (0, 5),
    "LIMMTOTAL": (0, 25),
    "GDTOTAL": (0, 15),
    "FAQTOTAL": (0, 30),
    "TOTAL13": (0, 85),
}

DEMOGRAPHIC_FEATURES = [
    "entry_age", "sex_label", "education_baseline", "APOE4_dosage"
]

# LDELTOTAL is the primary memory feature. LIMMTOTAL is retained in the
# full table as a sensitivity alternative because the two are redundant.
PRIMARY_CLINICAL_FEATURES = [
    "CDRSB", "LDELTOTAL", "MMSCORE", "GDTOTAL"
]

PRIMARY_COMBINED_FEATURES = [
    *DEMOGRAPHIC_FEATURES, *PRIMARY_CLINICAL_FEATURES
]

PRIMARY_FEATURE_SOURCE_MAP = {
    "entry_age": "entry_age",
    "sex_label": "PTGENDER",
    "education_baseline": "PTEDUCAT",
    "APOE4_dosage": "GENOTYPE",
    **{feature: feature for feature in PRIMARY_CLINICAL_FEATURES},
}

PRIMARY_NUMERIC_STATIC_SOURCES = ["PTGENDER", "entry_age", "PTEDUCAT"]


def _first_non_null(series):
    values = series.dropna()
    return values.iloc[0] if not values.empty else np.nan


def _unique_or_missing(series):
    values = pd.unique(series.dropna())
    return values[0] if len(values) == 1 else np.nan


def _normalize_visit(series):
    visit = series.astype("string").str.strip().str.lower()
    return visit.replace({"": pd.NA, "m6": "m06"})


def _normalize_genotype(series):
    genotype = series.astype("string").str.strip().str.upper()
    genotype = genotype.str.replace("APOE", "", regex=False)
    genotype = genotype.str.replace("Ε", "", regex=False)
    genotype = genotype.str.replace("E", "", regex=False)
    allele_lists = genotype.str.findall(r"[234]")

    def canonicalize(alleles):
        if not isinstance(alleles, list) or len(alleles) != 2:
            return pd.NA
        ordered = sorted(alleles, key=int)
        return f"{ordered[0]}/{ordered[1]}"

    return allele_lists.map(canonicalize).astype("string")


def adjust_pvalues(p_values, method):
    # Benjamini-Hochberg FDR or Holm family-wise adjustment.
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(p_values.shape, np.nan, dtype=float)
    valid_mask = ~np.isnan(p_values)
    valid = p_values[valid_mask]
    if valid.size == 0:
        return adjusted
    if np.any((valid < 0) | (valid > 1)):
        raise ValueError("p-values must be between 0 and 1")

    order = np.argsort(valid)
    ordered = valid[order]
    m = len(ordered)

    if method == "fdr_bh":
        ordered_adjusted = ordered * m / np.arange(1, m + 1)
        ordered_adjusted = np.minimum.accumulate(ordered_adjusted[::-1])[::-1]
    elif method == "holm":
        ordered_adjusted = ordered * (m - np.arange(m))
        ordered_adjusted = np.maximum.accumulate(ordered_adjusted)
    else:
        raise ValueError("method must be 'fdr_bh' or 'holm'")

    restored = np.empty(m, dtype=float)
    restored[order] = np.clip(ordered_adjusted, 0, 1)
    adjusted[valid_mask] = restored
    return adjusted


def _static_conflict_audit(clean):
    columns = [
        "PTGENDER", "entry_age", "entry_date", "entry_group_5",
        "entry_group_3", "GENOTYPE", "PTEDUCAT",
    ]
    records = []
    for column in columns:
        counts = clean.groupby("subject_id", dropna=True)[column].nunique(dropna=True)
        for subject_id, n_unique in counts[counts > 1].items():
            observed = clean.loc[
                clean["subject_id"].eq(subject_id), column
            ].dropna().astype(str).unique().tolist()
            records.append({
                "subject_id": subject_id,
                "variable": column,
                "n_unique": int(n_unique),
                "observed_values": " | ".join(observed),
            })
    return pd.DataFrame(
        records,
        columns=["subject_id", "variable", "n_unique", "observed_values"],
    )


def _build_subject_static(clean):
    priority = clean["visit"].map({"bl": 0, "sc": 1, "scmri": 2}).fillna(3)
    ordered = (
        clean.assign(_priority=priority, _row=np.arange(len(clean)))
        .sort_values(["subject_id", "_priority", "_row"])
    )

    static_columns = [
        "PTGENDER", "entry_age", "entry_date", "entry_group_5",
        "entry_group_3", "GENOTYPE",
    ]
    subject_static = (
        ordered.groupby("subject_id", sort=False, dropna=True)[static_columns]
        .agg(_unique_or_missing)
        .reset_index()
    )

    education = (
        ordered.groupby("subject_id", sort=False, dropna=True)["PTEDUCAT"]
        .agg(_unique_or_missing)
        .rename("education_baseline")
        .reset_index()
    )
    subject_static = subject_static.merge(education, on="subject_id", how="left")

    conflict_counts = clean.groupby("subject_id", dropna=True)[
        static_columns + ["PTEDUCAT"]
    ].nunique(dropna=True)
    critical_conflict_columns = [
        "PTGENDER", "entry_age", "entry_group_5",
        "entry_group_3", "GENOTYPE", "PTEDUCAT",
    ]
    conflict_flags = pd.DataFrame({
        "subject_id": conflict_counts.index,
        "static_conflict_any": conflict_counts.gt(1).any(axis=1).to_numpy(),
        "entry_group_conflict": conflict_counts["entry_group_5"].gt(1).to_numpy(),
        "critical_static_conflict": (
            conflict_counts[critical_conflict_columns].gt(1).any(axis=1).to_numpy()
        ),
        "education_conflict": conflict_counts["PTEDUCAT"].gt(1).to_numpy(),
    })
    subject_static = subject_static.merge(conflict_flags, on="subject_id", how="left")
    unmapped_entry_subjects = set(
        clean.loc[clean["entry_group_unmapped"], "subject_id"].dropna()
    )
    subject_static["entry_group_unmapped_any"] = (
        subject_static["subject_id"].isin(unmapped_entry_subjects)
    )

    subject_static["sex_label"] = subject_static["PTGENDER"].map(
        {1.0: "Male", 2.0: "Female"}
    )
    subject_static["APOE4_dosage"] = (
        subject_static["GENOTYPE"].map(APOE4_DOSAGE_MAP).astype("Int64")
    )
    subject_static["APOE4_carrier"] = (
        subject_static["APOE4_dosage"].gt(0).astype("Int64")
    )
    subject_static.loc[
        subject_static["APOE4_dosage"].isna(), "APOE4_carrier"
    ] = pd.NA
    return subject_static


def _baseline_conflict_audit(clean):
    baseline_rows = clean[clean["visit"].isin(["sc", "bl"])]
    records = []
    for (subject_id, visit), group in baseline_rows.groupby(
        ["subject_id", "visit"], dropna=False
    ):
        for column in CLINICAL_COLS:
            values = group[column].dropna().unique()
            if len(values) > 1:
                records.append({
                    "subject_id": subject_id,
                    "visit": visit,
                    "variable": column,
                    "n_unique": len(values),
                    "observed_values": " | ".join(map(str, values)),
                })
    return pd.DataFrame(
        records,
        columns=[
            "subject_id", "visit", "variable", "n_unique", "observed_values"
        ],
    )


def _baseline_value(group, column):
    for visit in ("bl", "sc"):
        values = group.loc[group["visit"].eq(visit), column].dropna().unique()
        if len(values) == 1:
            return values[0], visit
        if len(values) > 1:
            return np.nan, f"conflict_{visit}"
    return np.nan, pd.NA


def _build_baseline_snapshot(clean, subject_static):
    baseline_rows = clean[clean["visit"].isin(["sc", "bl"])]
    records = []
    for subject_id, group in baseline_rows.groupby("subject_id", sort=False):
        record = {"subject_id": subject_id}
        for column in CLINICAL_COLS:
            value, source = _baseline_value(group, column)
            record[column] = value
            record[f"source_{column}"] = source
        records.append(record)

    baseline_clinical = pd.DataFrame(records)
    if baseline_clinical.empty:
        baseline_clinical = pd.DataFrame(
            columns=[
                "subject_id",
                *CLINICAL_COLS,
                *[f"source_{column}" for column in CLINICAL_COLS],
            ]
        )
    baseline_all = subject_static.merge(
        baseline_clinical, on="subject_id", how="left"
    )
    baseline_all["has_sc_or_bl"] = baseline_all["subject_id"].isin(
        baseline_rows["subject_id"]
    )
    baseline_all["has_any_baseline_clinical"] = (
        baseline_all[CLINICAL_COLS].notna().any(axis=1)
    )
    source_columns = [f"source_{column}" for column in CLINICAL_COLS]
    baseline_all["baseline_conflict_any"] = (
        baseline_all[source_columns]
        .fillna("")
        .astype(str)
        .apply(lambda column: column.str.startswith("conflict_"))
        .any(axis=1)
    )
    return baseline_all


def _score_range_audit(table):
    records = []
    for column, (minimum, maximum) in SCORE_LIMITS.items():
        values = pd.to_numeric(table[column], errors="coerce")
        records.append({
            "variable": column,
            "expected_min": minimum,
            "expected_max": maximum,
            "observed_min": values.min(),
            "observed_max": values.max(),
            "n_below": int(values.lt(minimum).sum()),
            "n_above": int(values.gt(maximum).sum()),
        })
    return pd.DataFrame(records)


def _duplicate_key_audit(table):
    mask = table.duplicated(["subject_id", "visit"], keep=False)
    columns = ["subject_id", "visit", "DIAGNOSIS", *CLINICAL_COLS]
    return table.loc[mask, columns].sort_values(["subject_id", "visit"])


def _model_feature_availability(table, input_columns):
    records = []
    for feature in PRIMARY_COMBINED_FEATURES:
        source_column = PRIMARY_FEATURE_SOURCE_MAP[feature]
        source_present = source_column in input_columns
        n_non_missing = int(table[feature].notna().sum())
        records.append({
            "feature": feature,
            "source_column": source_column,
            "source_present": source_present,
            "n_non_missing": n_non_missing,
            "gate_pass": bool(source_present and n_non_missing > 0),
        })
    return pd.DataFrame(records)


def preprocess_adni(df_raw):
    # Return cleaned longitudinal, baseline, model-input, and QC tables.
    missing_core = sorted(set(CORE_REQUIRED_COLUMNS) - set(df_raw.columns))
    if missing_core:
        raise ValueError(f"Missing required columns: {missing_core}")

    missing_expected = sorted(set(EXPECTED_COLUMNS) - set(df_raw.columns))
    clean = df_raw.copy(deep=True)
    clean["_source_row_id"] = np.arange(len(clean), dtype=int)
    for column in missing_expected:
        clean[column] = np.nan
    clean = clean.replace(
        ["", "NA", "N/A", "nan", "None", "NULL", "null"], np.nan
    )

    clean["subject_id"] = clean["subject_id"].astype("string").str.strip()
    clean.loc[clean["subject_id"].eq(""), "subject_id"] = pd.NA
    clean["visit"] = _normalize_visit(clean["visit"])
    clean["GENOTYPE_raw"] = clean["GENOTYPE"].astype("string").str.strip()

    if "EXAMDATE" in clean.columns:
        examdate_raw = clean["EXAMDATE"].copy()
        clean["exam_date"] = pd.to_datetime(
            examdate_raw, errors="coerce", format="mixed"
        )
        invalid_date_mask = examdate_raw.notna() & clean["exam_date"].isna()
        date_coercion_audit = pd.DataFrame({
            "_source_row_id": clean.loc[invalid_date_mask, "_source_row_id"],
            "subject_id": clean.loc[invalid_date_mask, "subject_id"],
            "visit": clean.loc[invalid_date_mask, "visit"],
            "raw_value": examdate_raw.loc[invalid_date_mask],
        }).reset_index(drop=True)
    else:
        date_coercion_audit = pd.DataFrame(
            columns=["_source_row_id", "subject_id", "visit", "raw_value"]
        )

    coercion_records = []
    for column in NUMERIC_COLS:
        original_values = clean[column].copy()
        converted = pd.to_numeric(original_values, errors="coerce")
        invalid_mask = original_values.notna() & converted.isna()
        if invalid_mask.any():
            for row_index in clean.index[invalid_mask]:
                coercion_records.append({
                    "_source_row_id": clean.at[row_index, "_source_row_id"],
                    "subject_id": clean.at[row_index, "subject_id"],
                    "visit": clean.at[row_index, "visit"],
                    "variable": column,
                    "raw_value": original_values.at[row_index],
                })
        clean[column] = converted
    numeric_coercion_audit = pd.DataFrame(
        coercion_records,
        columns=[
            "_source_row_id", "subject_id", "visit", "variable", "raw_value"
        ],
    )

    # Common ADNI numeric missing sentinels. Replace only in fields where
    # negative values are impossible; zero remains a valid score.
    sentinel_counts = {
        column: int(clean[column].isin([-4, -1]).sum())
        for column in NUMERIC_COLS
    }
    for column in NUMERIC_COLS:
        clean.loc[clean[column].isin([-4, -1]), column] = np.nan

    clean["GENOTYPE"] = _normalize_genotype(clean["GENOTYPE"])
    unmapped_genotype_mask = (
        clean["GENOTYPE_raw"].notna() & clean["GENOTYPE"].isna()
    )
    unmapped_genotypes = clean.loc[
        unmapped_genotype_mask,
        ["_source_row_id", "subject_id", "visit", "GENOTYPE_raw"],
    ].rename(columns={"GENOTYPE_raw": "raw_value"}).reset_index(drop=True)

    clean["entry_group_raw"] = (
        clean["entry_research_group"].astype("string").str.strip().str.upper()
    )
    harmonized_entry_group = clean["entry_group_raw"].replace({"SMC": "CN"})
    clean["entry_group_5"] = harmonized_entry_group.where(
        harmonized_entry_group.isin(ENTRY_GROUP_5_LEVELS)
    )
    clean["entry_group_3"] = clean["entry_group_5"].replace(
        {"EMCI": "MCI", "LMCI": "MCI"}
    )

    unmapped_entry_mask = (
        clean["entry_group_raw"].notna()
        & ~clean["entry_group_raw"].isin(ENTRY_GROUP_RAW_LEVELS)
    )
    clean["entry_group_unmapped"] = unmapped_entry_mask
    unmapped_entry_groups = clean.loc[
        unmapped_entry_mask,
        ["_source_row_id", "subject_id", "visit", "entry_group_raw"],
    ].rename(columns={"entry_group_raw": "raw_value"}).reset_index(drop=True)
    entry_group_mapping = (
        clean.groupby(
            ["entry_group_raw", "entry_group_5", "entry_group_3"],
            dropna=False,
        )
        .size()
        .reset_index(name="n_rows")
    )

    clean["diagnosis_qc_label"] = clean["DIAGNOSIS"].map(DIAGNOSIS_QC_MAP)
    clean["diagnosis_label"] = clean["diagnosis_qc_label"].where(
        clean["diagnosis_qc_label"].ne("TEAM_NODX")
    )
    unmapped_diagnosis_mask = (
        clean["DIAGNOSIS"].notna()
        & ~clean["DIAGNOSIS"].isin(DIAGNOSIS_QC_MAP)
    )
    unmapped_diagnoses = clean.loc[
        unmapped_diagnosis_mask,
        ["_source_row_id", "subject_id", "visit", "DIAGNOSIS"],
    ].rename(columns={"DIAGNOSIS": "raw_value"}).reset_index(drop=True)

    extracted_month = clean["visit"].str.extract(r"^m(\d+)$", expand=False)
    clean["visit_month"] = pd.to_numeric(
        extracted_month, errors="coerce"
    ).astype("Int64")
    clean.loc[clean["visit"].eq("bl"), "visit_month"] = 0
    clean["is_month_grid_visit"] = (
        clean["visit"].eq("bl") | clean["visit"].str.match(r"^m\d+$", na=False)
    )
    clean["has_any_clinical"] = clean[CLINICAL_COLS].notna().any(axis=1)

    # Never remove records automatically. Distinct source records can
    # become identical after column selection, so duplicates are audited
    # while the original row identifier and available timing/phase fields
    # remain attached.
    exact_duplicate_count = int(df_raw.duplicated().sum())
    clean = clean.reset_index(drop=True)

    missing_subject_rows = clean[clean["subject_id"].isna()].copy()
    missing_visit_rows = clean[
        clean["subject_id"].notna() & clean["visit"].isna()
    ].copy()
    clean = clean[clean["subject_id"].notna()].copy()

    static_conflicts = _static_conflict_audit(clean)
    subject_static = _build_subject_static(clean)

    retained_metadata = [
        column for column in OPTIONAL_RECORD_METADATA if column in clean.columns
    ]
    if "exam_date" in clean.columns:
        retained_metadata.append("exam_date")
    visit_columns = [
        "_source_row_id", "subject_id", "visit", *retained_metadata,
        "visit_month", "is_month_grid_visit",
        "has_any_clinical", "DIAGNOSIS", "diagnosis_qc_label",
        "diagnosis_label", *CLINICAL_COLS,
    ]
    clinical_long_clean = clean.loc[
        clean["visit"].notna(), visit_columns
    ].merge(subject_static, on="subject_id", how="left")

    phase_key = next(
        (
            column for column in ["Phase", "PHASE", "COLPROT", "ORIGPROT"]
            if column in clinical_long_clean.columns
        ),
        None,
    )
    visit_key = (
        "VISCODE2" if "VISCODE2" in clinical_long_clean.columns else "visit"
    )
    longitudinal_key_columns = ["subject_id", visit_key]
    if phase_key is not None:
        longitudinal_key_columns.append(phase_key)
    if "exam_date" in clinical_long_clean.columns:
        longitudinal_key_columns.append("exam_date")

    clinical_long_clean["longitudinal_key_conflict"] = (
        clinical_long_clean.duplicated(
            longitudinal_key_columns, keep=False
        )
    )
    invalid_date_source_rows = set(date_coercion_audit["_source_row_id"])
    clinical_long_clean["longitudinal_date_invalid"] = (
        clinical_long_clean["_source_row_id"].isin(invalid_date_source_rows)
    )
    clinical_long_clean["longitudinal_key_issue"] = (
        clinical_long_clean["longitudinal_key_conflict"]
        | clinical_long_clean["longitudinal_date_invalid"]
    )
    audit_columns = ["_source_row_id", *longitudinal_key_columns]
    if "EXAMDATE" in clinical_long_clean.columns:
        audit_columns.append("EXAMDATE")
    longitudinal_key_audit = clinical_long_clean.loc[
        clinical_long_clean["longitudinal_key_issue"], audit_columns,
    ].copy()
    if not longitudinal_key_audit.empty:
        conflict_by_source = clinical_long_clean.set_index("_source_row_id")[
            "longitudinal_key_conflict"
        ]
        invalid_by_source = clinical_long_clean.set_index("_source_row_id")[
            "longitudinal_date_invalid"
        ]
        longitudinal_key_audit["issue"] = [
            "duplicate_key_and_invalid_date"
            if conflict_by_source.loc[source_id] and invalid_by_source.loc[source_id]
            else "duplicate_key"
            if conflict_by_source.loc[source_id]
            else "invalid_date"
            for source_id in longitudinal_key_audit["_source_row_id"]
        ]
    else:
        longitudinal_key_audit["issue"] = pd.Series(dtype="string")

    duplicate_subject_visits = _duplicate_key_audit(clinical_long_clean)
    baseline_conflicts = _baseline_conflict_audit(clean)
    baseline_all = _build_baseline_snapshot(clean, subject_static)
    score_range_audit = _score_range_audit(baseline_all)

    range_flags = pd.DataFrame(index=baseline_all.index)
    for column, (minimum, maximum) in SCORE_LIMITS.items():
        range_flags[column] = (
            baseline_all[column].lt(minimum)
            | baseline_all[column].gt(maximum)
        )
    baseline_all["score_range_flag_any"] = range_flags.any(axis=1)
    baseline_all["primary_score_range_flag_any"] = range_flags[
        PRIMARY_CLINICAL_FEATURES
    ].any(axis=1)

    primary_source_columns = [
        f"source_{column}" for column in PRIMARY_CLINICAL_FEATURES
    ]
    baseline_all["primary_baseline_conflict_any"] = (
        baseline_all[primary_source_columns]
        .fillna("")
        .astype(str)
        .apply(lambda column: column.str.startswith("conflict_"))
        .any(axis=1)
    )

    primary_numeric_coercion_mask = (
        numeric_coercion_audit["variable"].isin(
            PRIMARY_NUMERIC_STATIC_SOURCES
        )
        | (
            numeric_coercion_audit["visit"].isin(["sc", "bl"])
            & numeric_coercion_audit["variable"].isin(
                PRIMARY_CLINICAL_FEATURES
            )
        )
    )
    baseline_bad_numeric_subjects = set(
        numeric_coercion_audit.loc[
            numeric_coercion_audit["visit"].isin(["sc", "bl"]),
            "subject_id",
        ].dropna()
    )
    primary_bad_numeric_subjects = set(
        numeric_coercion_audit.loc[
            primary_numeric_coercion_mask, "subject_id"
        ].dropna()
    )
    baseline_all["baseline_numeric_coercion_any"] = (
        baseline_all["subject_id"].isin(baseline_bad_numeric_subjects)
    )
    baseline_all["primary_numeric_coercion_any"] = (
        baseline_all["subject_id"].isin(primary_bad_numeric_subjects)
    )

    baseline_5 = baseline_all[
        baseline_all["entry_group_5"].isin(ENTRY_GROUP_5_LEVELS)
        & baseline_all["has_sc_or_bl"]
    ].copy()
    baseline_3 = baseline_all[
        baseline_all["entry_group_3"].isin(ENTRY_GROUP_3_LEVELS)
        & baseline_all["has_sc_or_bl"]
    ].copy()

    model_common = [
        "subject_id", "entry_group_5", "entry_group_3",
        *PRIMARY_COMBINED_FEATURES,
    ]
    model_eligible_3 = (
        baseline_3["has_sc_or_bl"]
        & ~baseline_3["critical_static_conflict"]
        & ~baseline_3["entry_group_conflict"]
        & ~baseline_3["entry_group_unmapped_any"]
        & ~baseline_3["primary_baseline_conflict_any"]
        & ~baseline_3["primary_score_range_flag_any"]
        & ~baseline_3["primary_numeric_coercion_any"]
    )
    model_eligible_5 = (
        baseline_5["has_sc_or_bl"]
        & ~baseline_5["critical_static_conflict"]
        & ~baseline_5["entry_group_conflict"]
        & ~baseline_5["entry_group_unmapped_any"]
        & ~baseline_5["primary_baseline_conflict_any"]
        & ~baseline_5["primary_score_range_flag_any"]
        & ~baseline_5["primary_numeric_coercion_any"]
    )
    baseline_model_3group = baseline_3.loc[model_eligible_3, model_common].copy()
    baseline_model_5group = baseline_5.loc[model_eligible_5, model_common].copy()
    model_feature_availability_3group = _model_feature_availability(
        baseline_model_3group, set(df_raw.columns)
    )
    model_feature_availability_5group = _model_feature_availability(
        baseline_model_5group, set(df_raw.columns)
    )

    preprocessing_summary = pd.DataFrame({
        "metric": [
            "raw_rows", "exact_duplicate_rows_flagged",
            "rows_missing_subject_id", "rows_missing_visit",
            "clean_long_rows", "clean_long_subjects",
            "baseline_subjects", "baseline_5group_subjects",
            "baseline_3group_subjects", "duplicate_subject_visit_rows",
            "static_conflicts", "baseline_value_conflicts",
            "numeric_missing_sentinels_replaced", "numeric_coercion_rows",
            "unmapped_entry_group_rows", "unmapped_genotype_rows",
            "unmapped_diagnosis_rows", "invalid_examdate_rows",
            "longitudinal_key_conflict_rows", "longitudinal_key_issue_rows",
            "baseline_model_5group_rows", "baseline_model_3group_rows",
            "baseline_model_5group_blocking_features",
            "baseline_model_3group_blocking_features",
        ],
        "value": [
            len(df_raw), exact_duplicate_count,
            len(missing_subject_rows), len(missing_visit_rows),
            len(clinical_long_clean), clinical_long_clean["subject_id"].nunique(),
            len(baseline_all), len(baseline_5), len(baseline_3),
            len(duplicate_subject_visits), len(static_conflicts),
            len(baseline_conflicts), sum(sentinel_counts.values()),
            len(numeric_coercion_audit), len(unmapped_entry_groups),
            len(unmapped_genotypes), len(unmapped_diagnoses),
            len(date_coercion_audit),
            int(clinical_long_clean["longitudinal_key_conflict"].sum()),
            int(clinical_long_clean["longitudinal_key_issue"].sum()),
            len(baseline_model_5group), len(baseline_model_3group),
            int((~model_feature_availability_5group["gate_pass"]).sum()),
            int((~model_feature_availability_3group["gate_pass"]).sum()),
        ],
    })

    subject_exclusion_parts = []
    exclusion_rules = {
        "no_sc_or_bl": ~baseline_all["has_sc_or_bl"],
        "missing_or_unmapped_entry_group": (
            baseline_all["entry_group_5"].isna()
            & ~baseline_all["entry_group_conflict"]
        ),
        "entry_group_conflict": baseline_all["entry_group_conflict"],
        "unmapped_entry_group": baseline_all["entry_group_unmapped_any"],
        "critical_static_conflict": baseline_all["critical_static_conflict"],
        "primary_baseline_clinical_conflict": (
            baseline_all["primary_baseline_conflict_any"]
        ),
        "primary_out_of_range_score": (
            baseline_all["primary_score_range_flag_any"]
        ),
        "primary_numeric_coercion": (
            baseline_all["primary_numeric_coercion_any"]
        ),
    }
    for reason, mask in exclusion_rules.items():
        if mask.any():
            subject_exclusion_parts.append(
                baseline_all.loc[mask, ["subject_id"]].assign(reason=reason)
            )
    duplicate_key_subjects = clinical_long_clean.loc[
        clinical_long_clean["longitudinal_key_conflict"], ["subject_id"]
    ].drop_duplicates()
    if not duplicate_key_subjects.empty:
        subject_exclusion_parts.append(
            duplicate_key_subjects.assign(reason="duplicate_longitudinal_key")
        )
    invalid_date_subjects = clinical_long_clean.loc[
        clinical_long_clean["longitudinal_date_invalid"], ["subject_id"]
    ].drop_duplicates()
    if not invalid_date_subjects.empty:
        subject_exclusion_parts.append(
            invalid_date_subjects.assign(reason="invalid_longitudinal_date")
        )
    subject_exclusions = (
        pd.concat(subject_exclusion_parts, ignore_index=True)
        if subject_exclusion_parts
        else pd.DataFrame(columns=["subject_id", "reason"])
    )

    release_status = pd.DataFrame([
        {
            "artifact": "clinical_long_clean",
            "ready_for_downstream": bool(
                ~clinical_long_clean["longitudinal_key_issue"].any()
            ),
            "blocking_records": int(
                clinical_long_clean["longitudinal_key_issue"].sum()
            ),
            "blocking_features": 0,
            "release_rule": (
                "no duplicate expanded keys or invalid non-null exam dates"
            ),
        },
        {
            "artifact": "baseline_model_5group",
            "ready_for_downstream": bool(
                len(baseline_model_5group) > 0
                and baseline_model_5group["subject_id"].is_unique
                and model_feature_availability_5group["gate_pass"].all()
            ),
            "blocking_records": int(
                len(baseline_all) - len(baseline_model_5group)
            ),
            "blocking_features": int(
                (~model_feature_availability_5group["gate_pass"]).sum()
            ),
            "release_rule": (
                "primary-feature schema passes and ambiguous/invalid "
                "primary-model records are quarantined"
            ),
        },
        {
            "artifact": "baseline_model_3group",
            "ready_for_downstream": bool(
                len(baseline_model_3group) > 0
                and baseline_model_3group["subject_id"].is_unique
                and model_feature_availability_3group["gate_pass"].all()
            ),
            "blocking_records": int(
                len(baseline_all) - len(baseline_model_3group)
            ),
            "blocking_features": int(
                (~model_feature_availability_3group["gate_pass"]).sum()
            ),
            "release_rule": (
                "primary-feature schema passes and ambiguous/invalid "
                "primary-model records are quarantined"
            ),
        },
    ])

    exclusions = pd.concat(
        [
            missing_subject_rows.assign(exclusion_reason="missing_subject_id"),
            missing_visit_rows.assign(exclusion_reason="missing_visit"),
        ],
        ignore_index=True,
    )

    return {
        "clinical_long_clean": clinical_long_clean,
        "subject_static": subject_static,
        "baseline_all": baseline_all,
        "baseline_5": baseline_5,
        "baseline_3": baseline_3,
        "baseline_model_3group": baseline_model_3group,
        "baseline_model_5group": baseline_model_5group,
        "preprocessing_summary": preprocessing_summary,
        "exclusions": exclusions,
        "duplicate_subject_visits": duplicate_subject_visits,
        "static_conflicts": static_conflicts,
        "baseline_conflicts": baseline_conflicts,
        "score_range_audit": score_range_audit,
        "entry_group_mapping": entry_group_mapping,
        "unmapped_entry_groups": unmapped_entry_groups,
        "unmapped_genotypes": unmapped_genotypes,
        "unmapped_diagnoses": unmapped_diagnoses,
        "numeric_coercion_audit": numeric_coercion_audit,
        "date_coercion_audit": date_coercion_audit,
        "longitudinal_key_columns": longitudinal_key_columns,
        "longitudinal_key_audit": longitudinal_key_audit,
        "subject_exclusions": subject_exclusions,
        "release_status": release_status,
        "model_feature_availability_3group": (
            model_feature_availability_3group
        ),
        "model_feature_availability_5group": (
            model_feature_availability_5group
        ),
        "missing_input_columns": pd.DataFrame(
            {"column": missing_expected, "action": "inserted_as_missing"}
        ),
    }


## 3. Run preprocessing and inspect the audit

A nonzero conflict count is not an instruction to average records. Inspect the corresponding audit table and reconcile the source data or add the missing visit/phase key before proceeding.


In [ ]:
processed = preprocess_adni(df_raw)

clinical_long_clean = processed["clinical_long_clean"]
subject_static = processed["subject_static"]
baseline_all = processed["baseline_all"]
baseline_5 = processed["baseline_5"]
baseline_3 = processed["baseline_3"]
baseline_model_3group = processed["baseline_model_3group"]
baseline_model_5group = processed["baseline_model_5group"]

display(processed["preprocessing_summary"])


In [ ]:
# The preprocessing function must not mutate the input dataframe.
pd.testing.assert_frame_equal(df_raw, df_raw_snapshot)

audit_tables = {
    "rows excluded from visit-level table": processed["exclusions"],
    "subject-level eligibility/exclusion reasons": processed["subject_exclusions"],
    "duplicate subject–visit rows": processed["duplicate_subject_visits"],
    "unresolved expanded longitudinal keys": processed["longitudinal_key_audit"],
    "subject-level static conflicts": processed["static_conflicts"],
    "baseline clinical conflicts": processed["baseline_conflicts"],
    "missing expected input columns": processed["missing_input_columns"],
    "numeric values coerced to missing": processed["numeric_coercion_audit"],
    "dates coerced to missing": processed["date_coercion_audit"],
    "unmapped entry groups": processed["unmapped_entry_groups"],
    "unmapped APOE genotypes": processed["unmapped_genotypes"],
    "unmapped diagnosis codes": processed["unmapped_diagnoses"],
}

for name, table in audit_tables.items():
    print(f"{name}: {len(table):,}")
    if not table.empty:
        display(table.head(20))

print("Expanded longitudinal key:", processed["longitudinal_key_columns"])
display(processed["entry_group_mapping"])
display(processed["model_feature_availability_3group"])
display(processed["model_feature_availability_5group"])
display(processed["release_status"])


In [ ]:
display(processed["score_range_audit"])

n_out_of_range = (
    processed["score_range_audit"]["n_below"]
    + processed["score_range_audit"]["n_above"]
).sum()
print(f"Total values outside the stated QC ranges: {n_out_of_range:,}")


## 4. Visit structure and clinical availability

Missing clinical values are largely protocol-driven. Therefore, this notebook does not perform global row deletion. `scmri` is preserved as a possible future MRI-alignment anchor even when it contains no clinical score.


In [ ]:
visit_summary = (
    clinical_long_clean.groupby("visit", dropna=False)
    .agg(
        n_rows=("subject_id", "size"),
        n_subjects=("subject_id", "nunique"),
        pct_any_clinical=("has_any_clinical", lambda x: 100 * x.mean()),
    )
)

clinical_availability = (
    clinical_long_clean.groupby("visit")[CLINICAL_COLS]
    .apply(lambda frame: 100 * frame.notna().mean())
)

visit_clinical_summary = (
    visit_summary.join(clinical_availability)
    .sort_values("n_subjects", ascending=False)
    .round(1)
)
display(visit_clinical_summary)


In [ ]:
top_visits = visit_clinical_summary.head(15).index
availability_plot = visit_clinical_summary.loc[top_visits, CLINICAL_COLS]

plt.figure(figsize=(11, 6))
sns.heatmap(
    availability_plot,
    cmap="Blues",
    vmin=0,
    vmax=100,
    annot=True,
    fmt=".0f",
    cbar_kws={"label": "% non-missing"},
)
plt.title("Clinical-score availability at the 15 most common visits")
plt.xlabel("Clinical variable")
plt.ylabel("Visit")
plt.tight_layout()
plt.show()
plt.close()


## 5. Baseline construction and provenance

`sc` and `bl` are complementary in this export. The baseline snapshot uses a variable-wise hierarchy:

1. Use the `bl` value when available.
2. Otherwise use the `sc` value.
3. Otherwise retain missingness.

The `source_*` variables make this auditable. They should remain metadata, not model predictors.


In [ ]:
source_cols = [f"source_{column}" for column in CLINICAL_COLS]
source_summary = pd.DataFrame({
    column: baseline_all[f"source_{column}"].value_counts(dropna=False)
    for column in CLINICAL_COLS
}).T.fillna(0).astype(int)

baseline_eligible = baseline_all[baseline_all["has_sc_or_bl"]]
baseline_missingness = pd.DataFrame({
    "n_missing": baseline_eligible[CLINICAL_COLS].isna().sum(),
    "pct_missing": 100 * baseline_eligible[CLINICAL_COLS].isna().mean(),
}).round(1)

display(source_summary)
display(baseline_missingness.sort_values("pct_missing", ascending=False))


In [ ]:
if baseline_5.empty and baseline_3.empty:
    print("No baseline-eligible labeled subjects; cohort plots skipped.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    if baseline_5.empty:
        axes[0].text(0.5, 0.5, "No eligible subjects", ha="center")
    else:
        sns.countplot(
            data=baseline_5,
            x="entry_group_5",
            order=ENTRY_GROUP_5_LEVELS,
            color="#4C78A8",
            ax=axes[0],
        )
    axes[0].set_title("Five entry cohorts")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Subjects")

    if baseline_3.empty:
        axes[1].text(0.5, 0.5, "No eligible subjects", ha="center")
    else:
        sns.countplot(
            data=baseline_3,
            x="entry_group_3",
            order=ENTRY_GROUP_3_LEVELS,
            color="#2A9D8F",
            ax=axes[1],
        )
    axes[1].set_title("Collapsed three-group cohort")
    axes[1].set_xlabel("")
    axes[1].set_ylabel("Subjects")

    plt.tight_layout()
    plt.show()
    plt.close(fig)


In [ ]:
# Compare entry cohort with the available visit-specific diagnosis.
diagnosis_entry_qc = pd.crosstab(
    clinical_long_clean["entry_group_3"],
    clinical_long_clean["diagnosis_qc_label"],
    margins=True,
)
display(diagnosis_entry_qc)

team_nodx_rows = clinical_long_clean[
    clinical_long_clean["diagnosis_qc_label"].eq("TEAM_NODX")
]
print(f"TEAM_NODX rows retained for QC: {len(team_nodx_rows):,}")


## 6. Missingness and model-set eligibility

These counts describe availability only. Do not create a global complete-case dataset yet: that could unnecessarily shrink the cohort and change group balance. Imputation must be learned within each training fold in the next notebook.


In [ ]:
def complete_case_summary(table, group_column):
    feature_sets = {
        "demographic": DEMOGRAPHIC_FEATURES,
        "clinical_benchmark": PRIMARY_CLINICAL_FEATURES,
        "combined": PRIMARY_COMBINED_FEATURES,
    }
    records = []
    for group, group_data in table.groupby(group_column, observed=True):
        record = {"group": group, "n_total": len(group_data)}
        for name, features in feature_sets.items():
            record[f"n_complete_{name}"] = int(
                group_data[features].notna().all(axis=1).sum()
            )
            record[f"pct_complete_{name}"] = round(
                100 * group_data[features].notna().all(axis=1).mean(), 1
            )
        records.append(record)
    return pd.DataFrame(records)


completeness_3group = complete_case_summary(baseline_3, "entry_group_3")
completeness_5group = complete_case_summary(baseline_5, "entry_group_5")

display(completeness_3group)
display(completeness_5group)


In [ ]:
if baseline_3.empty:
    missing_by_group = pd.DataFrame(
        columns=DEMOGRAPHIC_FEATURES + CLINICAL_COLS,
        index=ENTRY_GROUP_3_LEVELS,
        dtype=float,
    )
    print("No three-group baseline cohort; missingness heatmap skipped.")
else:
    missing_by_group = (
        baseline_3.groupby("entry_group_3", observed=True)[
            DEMOGRAPHIC_FEATURES + CLINICAL_COLS
        ]
        .apply(lambda frame: 100 * frame.isna().mean())
        .round(1)
        .reindex(ENTRY_GROUP_3_LEVELS)
        .astype(float)
    )

    plt.figure(figsize=(12, 4))
    sns.heatmap(
        missing_by_group,
        cmap="Reds",
        vmin=0,
        vmax=100,
        annot=True,
        fmt=".1f",
        cbar_kws={"label": "% missing"},
    )
    plt.title("Baseline missingness by three-group entry cohort")
    plt.xlabel("")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()
    plt.close()


## 7. Focused baseline EDA

The plots and tests below describe the baseline cohort; they are not the final machine-learning feature-selection procedure. Clinical instruments such as CDR-SB and MMSE contribute to clinical characterization, so strong separation is expected and must be reported as a **clinical benchmark**, not as independent diagnostic discovery.


In [ ]:
EDA_NUMERIC_COLS = [
    "entry_age", "education_baseline", "CDRSB", "LDELTOTAL",
    "MMSCORE", "LIMMTOTAL", "GDTOTAL",
]

def q1(series):
    return series.quantile(0.25)


def q3(series):
    return series.quantile(0.75)


descriptive_by_group = (
    baseline_3.groupby("entry_group_3", observed=True)[EDA_NUMERIC_COLS]
    .agg(["count", "mean", "std", "median", q1, q3])
    .reindex(ENTRY_GROUP_3_LEVELS)
)
display(descriptive_by_group)


In [ ]:
if baseline_3.empty:
    print("No three-group baseline cohort; distribution plots skipped.")
else:
    fig, axes = plt.subplots(3, 3, figsize=(12, 10))
    axes = axes.flatten()

    available_plot_columns = [
        column for column in EDA_NUMERIC_COLS
        if baseline_3[column].notna().any()
    ]

    plot_sample = baseline_3.sample(
        min(len(baseline_3), 1200), random_state=42
    )
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="vert: bool will be deprecated.*",
            category=PendingDeprecationWarning,
        )
        for axis, column in zip(axes, available_plot_columns):
            sns.boxplot(
                data=baseline_3,
                x="entry_group_3",
                y=column,
                order=ENTRY_GROUP_3_LEVELS,
                color="#8ECAE6",
                showfliers=False,
                ax=axis,
            )
            sns.stripplot(
                data=plot_sample,
                x="entry_group_3",
                y=column,
                order=ENTRY_GROUP_3_LEVELS,
                color="#203040",
                alpha=0.16,
                size=1.5,
                ax=axis,
            )
            axis.set_title(column)
            axis.set_xlabel("")
            axis.set_ylabel("")

    for axis in axes[len(available_plot_columns):]:
        axis.remove()

    if available_plot_columns:
        fig.suptitle("Baseline distributions by entry cohort", y=1.01)
        plt.tight_layout()
        plt.show()
        plt.close(fig)
    else:
        plt.close(fig)
        print("All EDA variables are missing; distribution plots skipped.")


In [ ]:
def cramers_v_test(table, categorical, target):
    contingency = pd.crosstab(table[categorical], table[target])
    if contingency.shape[0] < 2 or contingency.shape[1] < 2:
        return pd.Series({
            "variable": categorical,
            "chi2": np.nan,
            "dof": np.nan,
            "p_value": np.nan,
            "cramers_v": np.nan,
            "minimum_expected": np.nan,
            "pct_expected_below_5": np.nan,
        })
    chi2, p_value, dof, expected = chi2_contingency(contingency)
    n = contingency.to_numpy().sum()
    denominator = n * min(contingency.shape[0] - 1, contingency.shape[1] - 1)
    cramers_v = np.sqrt(chi2 / denominator) if denominator > 0 else np.nan
    return pd.Series({
        "variable": categorical,
        "chi2": chi2,
        "dof": dof,
        "p_value": p_value,
        "cramers_v": cramers_v,
        "minimum_expected": expected.min(),
        "pct_expected_below_5": 100 * (expected < 5).mean(),
    })


categorical_results = pd.DataFrame([
    cramers_v_test(baseline_3, "sex_label", "entry_group_3"),
    cramers_v_test(baseline_3, "APOE4_carrier", "entry_group_3"),
])
display(categorical_results)

apoe4_prevalence = 100 * pd.crosstab(
    baseline_3["entry_group_3"],
    baseline_3["APOE4_carrier"],
    normalize="index",
)
display(apoe4_prevalence.round(1))


In [ ]:
def kruskal_wallis_table(table, features, group_column, group_order):
    records = []
    for feature in features:
        samples = [
            table.loc[table[group_column].eq(group), feature].dropna()
            for group in group_order
        ]
        if any(len(sample) == 0 for sample in samples):
            continue
        h_statistic, p_value = kruskal(*samples)
        n = sum(len(sample) for sample in samples)
        k = len(samples)
        epsilon_squared = (
            max(0, (h_statistic - k + 1) / (n - k))
            if n > k else np.nan
        )
        records.append({
            "feature": feature,
            "n": n,
            "H_statistic": h_statistic,
            "p_value": p_value,
            "epsilon_squared": epsilon_squared,
        })
    result = pd.DataFrame(records)
    if not result.empty:
        result["p_FDR"] = adjust_pvalues(result["p_value"], method="fdr_bh")
        result = result.sort_values("epsilon_squared", ascending=False)
    return result


kw_results = kruskal_wallis_table(
    baseline_3, EDA_NUMERIC_COLS, "entry_group_3", ENTRY_GROUP_3_LEVELS
)
display(kw_results)


In [ ]:
def pairwise_mann_whitney(table, features, group_column, group_order):
    records = []
    for feature in features:
        feature_records = []
        for group_1, group_2 in combinations(group_order, 2):
            x = table.loc[table[group_column].eq(group_1), feature].dropna()
            y = table.loc[table[group_column].eq(group_2), feature].dropna()
            if len(x) == 0 or len(y) == 0:
                continue
            u_statistic, p_value = mannwhitneyu(x, y, alternative="two-sided")
            feature_records.append({
                "feature": feature,
                "comparison": f"{group_1} vs {group_2}",
                "U": u_statistic,
                "p_value": p_value,
            })
        if feature_records:
            adjusted = adjust_pvalues(
                [row["p_value"] for row in feature_records], method="holm"
            )
            for row, p_holm in zip(feature_records, adjusted):
                row["p_Holm_within_feature"] = p_holm
            records.extend(feature_records)
    return pd.DataFrame(records)


pairwise_results = pairwise_mann_whitney(
    baseline_3, EDA_NUMERIC_COLS, "entry_group_3", ENTRY_GROUP_3_LEVELS
)
display(pairwise_results)


## 8. Redundancy assessment and final feature sets

Use Spearman correlation because several clinical scores are skewed or ordinal-like. A high pairwise correlation is a redundancy warning, not an automatic deletion rule. Here, `LDELTOTAL` is predefined as the primary memory feature and `LIMMTOTAL` is retained only for a later sensitivity analysis.


In [ ]:
CORRELATION_COLS = [
    "entry_age", "education_baseline", "CDRSB", "LDELTOTAL",
    "MMSCORE", "LIMMTOTAL", "GDTOTAL",
]

correlation_eligible = [
    column for column in CORRELATION_COLS
    if baseline_3[column].notna().sum() >= 2
    and baseline_3[column].nunique(dropna=True) >= 2
]

if len(correlation_eligible) < 2:
    spearman_corr = pd.DataFrame(
        np.nan, index=CORRELATION_COLS, columns=CORRELATION_COLS
    )
    high_corr_pairs = pd.DataFrame(
        columns=["feature_1", "feature_2", "spearman_rho"]
    )
    print("Fewer than two usable varying variables; correlation plot skipped.")
else:
    plotted_corr = baseline_3[correlation_eligible].corr(method="spearman")
    spearman_corr = pd.DataFrame(
        np.nan, index=CORRELATION_COLS, columns=CORRELATION_COLS
    )
    spearman_corr.loc[correlation_eligible, correlation_eligible] = plotted_corr
    upper = plotted_corr.where(
        np.triu(np.ones(plotted_corr.shape), k=1).astype(bool)
    )
    high_corr_pairs = (
        upper.stack()
        .rename("spearman_rho")
        .reset_index()
        .rename(columns={"level_0": "feature_1", "level_1": "feature_2"})
    )
    high_corr_pairs = high_corr_pairs[
        high_corr_pairs["spearman_rho"].abs().ge(0.80)
    ].sort_values("spearman_rho", key=lambda x: x.abs(), ascending=False)

    mask = np.triu(np.ones_like(plotted_corr, dtype=bool))
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        plotted_corr,
        mask=mask,
        cmap="vlag",
        center=0,
        vmin=-1,
        vmax=1,
        annot=True,
        fmt=".2f",
        square=True,
    )
    plt.title("Spearman correlations among baseline variables")
    plt.tight_layout()
    plt.show()
    plt.close()

display(high_corr_pairs)


In [ ]:
feature_set_dictionary = pd.DataFrame({
    "feature_set": [
        "Demographic/background",
        "Clinical benchmark",
        "Combined",
        "Sensitivity substitution",
    ],
    "features": [
        ", ".join(DEMOGRAPHIC_FEATURES),
        ", ".join(PRIMARY_CLINICAL_FEATURES),
        ", ".join(PRIMARY_COMBINED_FEATURES),
        "Replace LDELTOTAL with LIMMTOTAL; do not include both in the primary model",
    ],
    "role": [
        "Non-cognitive reference model",
        "Expected clinical upper benchmark; interpret diagnosis leakage explicitly",
        "Incremental comparison against the demographic model",
        "Robustness check for memory-measure choice",
    ],
})
display(feature_set_dictionary)


## 9. Final validation and export

The model-input tables intentionally retain missing values and categorical strings. A model table is released only when every predefined predictor's source column exists and the cleaned cohort contains at least one usable value for that predictor. The next notebook must use a subject-level split and a fitted preprocessing pipeline so imputation, scaling, and encoding are learned from training data only.


In [ ]:
# Structural checks before export.
assert df_raw.equals(df_raw_snapshot), "The raw dataframe was modified."
assert baseline_all["subject_id"].is_unique, "Baseline table is not one row per subject."
assert baseline_3["entry_group_3"].isin(ENTRY_GROUP_3_LEVELS).all()
assert baseline_5["entry_group_5"].isin(ENTRY_GROUP_5_LEVELS).all()
assert "LIMMTOTAL" not in baseline_model_3group.columns
assert set(PRIMARY_COMBINED_FEATURES).issubset(baseline_model_3group.columns)
assert baseline_model_3group["subject_id"].is_unique
assert baseline_model_5group["subject_id"].is_unique

validation_summary = pd.DataFrame({
    "table": [
        "clinical_long_clean", "baseline_all", "baseline_5",
        "baseline_3", "baseline_model_5group", "baseline_model_3group",
    ],
    "rows": [
        len(clinical_long_clean), len(baseline_all), len(baseline_5),
        len(baseline_3), len(baseline_model_5group),
        len(baseline_model_3group),
    ],
    "unique_subjects": [
        clinical_long_clean["subject_id"].nunique(),
        baseline_all["subject_id"].nunique(),
        baseline_5["subject_id"].nunique(),
        baseline_3["subject_id"].nunique(),
        baseline_model_5group["subject_id"].nunique(),
        baseline_model_3group["subject_id"].nunique(),
    ],
})
display(validation_summary)


In [ ]:
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

exports = {
    "baseline_clinical_all.csv": baseline_all,
    "baseline_clinical_5group.csv": baseline_5,
    "baseline_clinical_3group.csv": baseline_3,
    "preprocessing_summary.csv": processed["preprocessing_summary"],
    "run_manifest.csv": run_manifest,
    "preprocessing_exclusions.csv": processed["exclusions"],
    "subject_exclusions.csv": processed["subject_exclusions"],
    "missing_input_columns.csv": processed["missing_input_columns"],
    "numeric_coercion_audit.csv": processed["numeric_coercion_audit"],
    "date_coercion_audit.csv": processed["date_coercion_audit"],
    "entry_group_mapping.csv": processed["entry_group_mapping"],
    "unmapped_entry_groups.csv": processed["unmapped_entry_groups"],
    "unmapped_genotypes.csv": processed["unmapped_genotypes"],
    "unmapped_diagnoses.csv": processed["unmapped_diagnoses"],
    "duplicate_subject_visits.csv": processed["duplicate_subject_visits"],
    "longitudinal_key_audit.csv": processed["longitudinal_key_audit"],
    "static_conflicts.csv": processed["static_conflicts"],
    "baseline_conflicts.csv": processed["baseline_conflicts"],
    "score_range_audit.csv": processed["score_range_audit"],
    "release_status.csv": processed["release_status"],
    "model_feature_availability_3group.csv": (
        processed["model_feature_availability_3group"]
    ),
    "model_feature_availability_5group.csv": (
        processed["model_feature_availability_5group"]
    ),
    "complete_case_summary_3group.csv": completeness_3group,
    "complete_case_summary_5group.csv": completeness_5group,
    "feature_set_dictionary.csv": feature_set_dictionary,
}

release_lookup = processed["release_status"].set_index("artifact")
if release_lookup.loc["clinical_long_clean", "ready_for_downstream"]:
    exports["clinical_long_clean.csv"] = clinical_long_clean
else:
    exports["clinical_long_REQUIRES_KEY_RESOLUTION.csv"] = clinical_long_clean
    print(
        "Longitudinal table was not released as clean: resolve the keys "
        "listed in longitudinal_key_audit.csv."
    )

if release_lookup.loc["baseline_model_5group", "ready_for_downstream"]:
    exports["baseline_model_input_5group.csv"] = baseline_model_5group
else:
    print("Five-group model input was not released; inspect release_status.csv.")

if release_lookup.loc["baseline_model_3group", "ready_for_downstream"]:
    exports["baseline_model_input_3group.csv"] = baseline_model_3group
else:
    print("Three-group model input was not released; inspect release_status.csv.")

for filename, table in exports.items():
    table.to_csv(RUN_OUTPUT_DIR / filename, index=False)

print(f"Exported {len(exports)} tables to {RUN_OUTPUT_DIR.resolve()}")


## Stop here: next notebook

After confirming `release_status.csv`, the next notebook should begin from `baseline_model_input_3group.csv` and compare:

1. Demographic/background model
2. Clinical benchmark model
3. Combined model

Use stratified **subject-level** cross-validation. Put imputation, one-hot encoding, scaling, and any data-driven feature selection inside the cross-validation pipeline. Keep the 5-group comparison exploratory. For the primary thesis direction, retain `clinical_long_clean.csv` to derive future score changes after a temporal horizon and later merge the baseline predictors with baseline FreeSurfer features.
